# 2.0 – Feature Extraction Testing

**Mục tiêu:**
- Kiểm tra và so sánh 2 phương pháp trích xuất đặc trưng: **PSD** và **DE**
- Trực quan hoá feature map trên toàn bộ 32 kênh / 4 dải tần
- Lưu đặc trưng đã xử lý ra thư mục `data/processed/`
- Benchmark thời gian trích xuất

In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath('..'))

import pickle
import time
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from src.preprocess import extract_features, BANDS

%matplotlib inline
plt.rcParams['figure.dpi'] = 100

DATA_DIR    = '../data/raw'
PROC_DIR    = '../data/processed'
os.makedirs(PROC_DIR, exist_ok=True)

## 1. Đọc dữ liệu một subject

In [ ]:
SUBJECT = 's01.dat'

with open(os.path.join(DATA_DIR, SUBJECT), 'rb') as f:
    sub = pickle.load(f, encoding='latin1')

data   = sub['data'][:, :32, :]   # (40 trials, 32 EEG ch, 8064 samples)
labels = sub['labels'][:, :2]     # valence, arousal

print('EEG data   :', data.shape)
print('Labels     :', labels.shape)

## 2. So sánh PSD vs DE – 1 trial

In [ ]:
trial = data[0]   # (32, 8064)

psd_feats = extract_features(trial, mode='psd')   # (32, 4)
de_feats  = extract_features(trial, mode='de')    # (32, 4)

band_names = list(BANDS.keys())

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
for ax, feats, title in zip(axes,
                             [psd_feats, de_feats],
                             ['PSD Features (32 ch × 4 bands)',
                              'DE  Features (32 ch × 4 bands)']):
    im = ax.imshow(feats, aspect='auto', cmap='YlOrRd')
    ax.set_title(title)
    ax.set_xlabel('Dải tần số')
    ax.set_ylabel('Kênh EEG')
    ax.set_xticks(range(4))
    ax.set_xticklabels(band_names, rotation=30)
    plt.colorbar(im, ax=ax)

plt.suptitle('Feature Map – Trial 0', fontsize=13)
plt.tight_layout()
plt.show()

## 3. Phân phối PSD theo dải tần (32 kênh)

In [ ]:
fig, axes = plt.subplots(1, 4, figsize=(14, 3), sharey=False)

for i, (band, ax) in enumerate(zip(band_names, axes)):
    ax.boxplot(psd_feats[:, i], vert=True)
    ax.set_title(band)
    ax.set_ylabel('PSD mean' if i == 0 else '')
    ax.set_xticks([])

plt.suptitle('PSD per band – tất cả 32 kênh (Trial 0)')
plt.tight_layout()
plt.show()

## 4. Benchmark: tốc độ trích xuất toàn bộ 40 trial

In [ ]:
for mode in ['psd', 'de']:
    start = time.perf_counter()
    all_feats = np.stack([extract_features(data[i], mode=mode) for i in range(40)])
    elapsed   = time.perf_counter() - start
    print(f'[{mode.upper()}] shape={all_feats.shape} | time={elapsed:.2f}s ({elapsed/40*1000:.1f}ms/trial)')

## 5. Lưu đặc trưng cho 1 subject

In [ ]:
psd_all = np.stack([extract_features(data[i], mode='psd') for i in range(40)])  # (40, 32, 4)
de_all  = np.stack([extract_features(data[i], mode='de')  for i in range(40)])

subj_id = os.path.splitext(SUBJECT)[0]   # 's01'
np.save(os.path.join(PROC_DIR, f'{subj_id}_psd.npy'), psd_all)
np.save(os.path.join(PROC_DIR, f'{subj_id}_de.npy'),  de_all)
np.save(os.path.join(PROC_DIR, f'{subj_id}_labels.npy'), labels)

print('Saved!')
print('  PSD  :', psd_all.shape)
print('  DE   :', de_all.shape)
print('  labels:', labels.shape)